# Grounding and Abstention by Design — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/rag-systems/5-4/grounding-and-abstention-by-design.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/rag-systems/5-4/grounding-and-abstention-by-design.ipynb)

Companion to: [Grounding and Abstention by Design](https://anacodicai.org/learn/rag-systems/5-4/grounding-and-abstention-by-design)


## What you'll build

1. A small corpus and a mix of in-scope / out-of-scope questions.
2. A **naive** answerer that always generates a confident-sounding answer.
3. A **grounded** answerer that checks the retrieval score BEFORE generating —
   and correctly refuses when nothing supports an answer.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
import re
import math
from collections import Counter
print("ready — standard library only")

ready — standard library only


## Step 1 — a small corpus, and questions marked in/out of scope

**Try it:** add your own out-of-scope question — something the corpus below
clearly doesn't cover — and see whether it correctly gets refused later.

In [2]:
CORPUS = {
    "d1": "The recycling program accepts glass, aluminum cans, and cardboard "
          "every Tuesday morning starting at 7am.",
    "d2": "Parking permits for residents cost 40 dollars a year and can be "
          "renewed online or at the town office.",
    "d3": "Noise complaints after 10pm on weekdays can be reported through "
          "the town's non-emergency line.",
}

QUESTIONS = [
    ("When does the recycling program pick up glass and cardboard?", True),
    ("How much does a resident parking permit cost per year?", True),
    ("How do I report a noise complaint after 10pm?", True),
    ("What is the property tax rate for a house in this town?", False),
    ("Is street parking free on national holidays?", False),
]
print(f"{len(CORPUS)} documents, {len(QUESTIONS)} test questions")

3 documents, 5 test questions


## Step 2 — retrieval

Plain stemmed cosine similarity, same technique as the chunking notebook.

In [3]:
STOPWORDS = {"the", "a", "an", "is", "does", "do", "how", "what", "on",
             "for", "in", "at", "to", "and", "this", "i", "my", "if"}

def vectorize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return Counter(w for w in words if w not in STOPWORDS)

def cosine(a, b):
    common = set(a) & set(b)
    dot = sum(a[w] * b[w] for w in common)
    mag_a = math.sqrt(sum(v * v for v in a.values()))
    mag_b = math.sqrt(sum(v * v for v in b.values()))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

DOC_VECS = {k: vectorize(v) for k, v in CORPUS.items()}

def best_match(question):
    q_vec = vectorize(question)
    scored = sorted(((cosine(q_vec, v), k) for k, v in DOC_VECS.items()), reverse=True)
    return scored[0]

print(f"{'in-scope?':10}{'question':52}{'best score':11}")
for question, in_scope in QUESTIONS:
    score, doc_id = best_match(question)
    print(f"{str(in_scope):10}{question:52}{score:<11.3f}")

in-scope? question                                            best score 
True      When does the recycling program pick up glass and cardboard?0.436      
True      How much does a resident parking permit cost per year?0.314      
True      How do I report a noise complaint after 10pm?       0.359      
False     What is the property tax rate for a house in this town?0.124      
False     Is street parking free on national holidays?        0.124      


## Step 3 — the naive answerer

Always generates a confident-sounding answer, regardless of how weak the
match actually is.

In [4]:
def answer_naive(question):
    score, doc_id = best_match(question)
    return f"Based on {doc_id}: [a confident-sounding answer, generated " \
           f"regardless of score={score:.2f}]"

for question, in_scope in QUESTIONS:
    if not in_scope:
        print(f"Q: {question}\n-> {answer_naive(question)}\n")

Q: What is the property tax rate for a house in this town?
-> Based on d2: [a confident-sounding answer, generated regardless of score=0.12]

Q: Is street parking free on national holidays?
-> Based on d2: [a confident-sounding answer, generated regardless of score=0.12]



## Step 4 — the grounded answerer

**Try it:** lower `GROUNDING_THRESHOLD` toward `0.0` and re-run — watch it
start (wrongly) answering out-of-scope questions again. Raise it too high and
it will start (wrongly) refusing genuinely in-scope ones.

In [5]:
GROUNDING_THRESHOLD = 0.15

def answer_grounded(question):
    score, doc_id = best_match(question)
    if score < GROUNDING_THRESHOLD:
        return f"Not found in the source material (best match {doc_id}, " \
               f"score={score:.2f} below threshold {GROUNDING_THRESHOLD})"
    return f"Based on {doc_id} (score={score:.2f}): [a real answer, " \
           f"generated because retrieval cleared the threshold]"

for question, in_scope in QUESTIONS:
    print(f"[{'in-scope' if in_scope else 'OUT-of-scope':13}] {question}")
    print(f"   -> {answer_grounded(question)}\n")

[in-scope     ] When does the recycling program pick up glass and cardboard?
   -> Based on d1 (score=0.44): [a real answer, generated because retrieval cleared the threshold]

[in-scope     ] How much does a resident parking permit cost per year?
   -> Based on d2 (score=0.31): [a real answer, generated because retrieval cleared the threshold]

[in-scope     ] How do I report a noise complaint after 10pm?
   -> Based on d3 (score=0.36): [a real answer, generated because retrieval cleared the threshold]

[OUT-of-scope ] What is the property tax rate for a house in this town?
   -> Not found in the source material (best match d2, score=0.12 below threshold 0.15)

[OUT-of-scope ] Is street parking free on national holidays?
   -> Not found in the source material (best match d2, score=0.12 below threshold 0.15)



## Step 5 — measure both designs against every question

In [6]:
naive_confident_on_oos = 0
grounded_correct_abstentions = 0
in_scope_grounded_answers = 0
oos_count = in_scope_count = 0

for question, in_scope in QUESTIONS:
    if in_scope:
        in_scope_count += 1
        if "real answer" in answer_grounded(question):
            in_scope_grounded_answers += 1
    else:
        oos_count += 1
        if "confident-sounding" in answer_naive(question):
            naive_confident_on_oos += 1
        if "Not found" in answer_grounded(question):
            grounded_correct_abstentions += 1

print(f"naive design: confidently answered {naive_confident_on_oos}/{oos_count} "
      f"out-of-scope questions anyway")
print(f"grounded design: correctly abstained on {grounded_correct_abstentions}/{oos_count} "
      f"out-of-scope questions")
print(f"grounded design: still answered {in_scope_grounded_answers}/{in_scope_count} "
      f"in-scope questions normally")

assert naive_confident_on_oos == oos_count
assert grounded_correct_abstentions == oos_count
assert in_scope_grounded_answers == in_scope_count
print("\nall asserts passed")

naive design: confidently answered 2/2 out-of-scope questions anyway
grounded design: correctly abstained on 2/2 out-of-scope questions
grounded design: still answered 3/3 in-scope questions normally

all asserts passed


## Try it yourself

- Add a question that's borderline — related to the corpus but not directly
  answered by it — and watch its score land between the in-scope and
  out-of-scope clusters. That's the "no threshold cleanly separates everything"
  case the article's §8 and §4 warn about.
- Rephrase one in-scope question using totally different vocabulary from its
  source document (e.g. describing "recycling" as "trash sorting drop-off")
  and see the score — and possibly a false abstention — change.